# Precision Trading System — Full-Scale Training Notebook

**Pipeline:** smoke test → data quality check → gate → walk-forward CV → stacking ensemble → calibration → evaluation → download.

**Stack:** CatBoost + LightGBM + XGBoost stacking ensemble + isotonic calibration + meta-labeling.

**Key Research Foundations:**
- **Triple-Barrier Labeling** (López de Prado, 2018) — pt/sl/time barriers eliminate look-ahead bias
- **Purged Walk-Forward CV** (López de Prado, 2018) — prevents label overlap leakage in time-series
- **Meta-Labeling** (López de Prado, 2018) — secondary model filters false positives
- **Stacked Generalization** (Wolpert, 1992; Breiman, 1996) — ensemble of heterogeneous classifiers
- **Isotonic Calibration** (Zadrozny & Elkan, 2002) — monotonic probability calibration
- **Markov Regime Detection** (Hamilton, 1989; Kritzman et al., 2012) — regime-aware feature augmentation
- **Microstructure Features** — VPIN (Easley et al., 2012), CVD, Volume Profile, Roll spread

## Sections
1. Environment setup (Colab-ready)
2. Data loading (4 sources: upload / yfinance / prebuilt CSV / Dukascopy)
3. Data quality & validation
4. Smoke test on 5–10k bars (gate)
5. Walk-forward cross-validation
6. Full training — CatBoost + LightGBM + XGBoost stacking
7. Calibration verification
8. Risk-adjusted performance evaluation
9. Comparison: stacked vs single model
10. Feature importance & explainability
11. Save + download model artifact
12. In-notebook inference test

## 1. Environment setup

Two paths — pick one. Path A clones your repo (preferred); Path B installs only the dependencies and you paste code inline.

In [ ]:
# ── Path A: clone repo (recommended) ──────────────────────────────
import os, sys, subprocess

REPO_URL = "https://github.com/YOUR_USERNAME/vibetrading.git"  # ← edit me
REPO_DIR = "/content/vibetrading"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

FRONTEND = os.path.join(REPO_DIR, "trading_terminal", "frontend")
sys.path.insert(0, FRONTEND)
os.chdir(FRONTEND)
print("Frontend on path:", FRONTEND)

In [ ]:
# ── Install dependencies ──
%pip install -q xgboost lightgbm catboost scikit-learn pandas numpy yfinance \
    pykalman hmmlearn joblib matplotlib seaborn scipy

In [ ]:
# Sanity check imports
import warnings; warnings.filterwarnings("ignore")
import os; os.environ["ENABLE_TRADING_MEMORY"] = "false"
import numpy as np, pandas as pd
import xgboost as xgb, lightgbm as lgb
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats

try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False
    print("CatBoost unavailable — falling back to XGBoost+LightGBM ensemble")

from services.precision_trading_system import (
    PrecisionTradingSystem, Asset, AssetConfig, ASSET_CONFIGS,
    triple_barrier_labels_vectorized,
    MarkovRegimeForecaster,
    ClassificationMetrics, BiasDetector,
    PurgedWalkForward, RiskManager, TradeDirection,
)
from services.training_pipeline import (
    TrainingConfig, TrainingPipeline, TrainingMetrics, TrainingStatus,
    DataCleaner, DataValidator, DriftDetector, FeatureEngineer,
    HyperparameterOptimizer, ModelRegistry,
)
from sklearn.metrics import (
    f1_score, matthews_corrcoef, accuracy_score,
    precision_score, recall_score, classification_report,
    confusion_matrix, roc_auc_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV

print("All imports OK · CatBoost:", HAS_CATBOOST)

## 2. Data loading

Pick ONE source. Cell 2A: upload your own CSV. Cell 2B: yfinance (limited to 30 days at 1m). Cell 2C: read the prebuilt `data/historical/<PAIR>_1m.csv`. Cell 2D: Dukascopy (FX/gold/CFDs, free historical 1m, requires `dukascopy_python`).

In [ ]:
# ── 2A: Upload your own OHLCV CSV ──
from google.colab import files
uploaded = files.upload()
csv_name = next(iter(uploaded.keys()))
df = pd.read_csv(csv_name)
if "timestamp" in df.columns:
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    df = df.set_index("timestamp")
df = df[["open", "high", "low", "close", "volume"]].sort_index().dropna()
print(f"Loaded {len(df):,} bars from {csv_name}")
print(df.head())

In [ ]:
# ── 2B: yfinance — last 30 days of 1m, or longer at 1h/1d ──
import yfinance as yf

YF_TICKER = "GC=F"     # gold futures (use BTC-USD / EURUSD=X / GBPUSD=X / etc.)
INTERVAL = "1m"
PERIOD = "30d"

df = yf.download(YF_TICKER, period=PERIOD, interval=INTERVAL,
                 progress=False, auto_adjust=True, threads=False)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
df.columns = [c.lower() for c in df.columns]
df = df[["open", "high", "low", "close", "volume"]].dropna()
df.index = pd.to_datetime(df.index, utc=True); df.index.name = "timestamp"
print(f"Loaded {len(df):,} bars · {df.index[0]} → {df.index[-1]}")

In [ ]:
# ── 2C: prebuilt CSV from the repo ──
PAIR = "XAUUSD"
TIMEFRAME = "1m"
csv = f"data/historical/{PAIR}_{TIMEFRAME}.csv"
df = pd.read_csv(csv, index_col="timestamp", parse_dates=["timestamp"])
if "source" in df.columns:
    df = df[df["source"] == "yfinance"]
df = df[["open", "high", "low", "close", "volume"]].sort_index().dropna()
print(f"Loaded {len(df):,} bars · {df.index[0]} → {df.index[-1]}")

In [ ]:
# ── 2D (optional): Dukascopy 1m for FX/gold (years of free history) ──
# %pip install -q dukascopy_python
# from dukascopy_python.instruments import INSTRUMENT_CFD_XAU_USD
# from dukascopy_python import fetch, INTERVAL_MIN_1, OFFER_SIDE_BID
# from datetime import datetime, timedelta
# end = datetime.now(); start = end - timedelta(days=540)
# df = fetch(INSTRUMENT_CFD_XAU_USD, INTERVAL_MIN_1, OFFER_SIDE_BID, start, end)
# df = df.rename(columns=str.lower)[["open","high","low","close","volume"]]
# df.index = pd.to_datetime(df.index, utc=True)
# df = df.resample("1min").agg({"open":"first","high":"max","low":"min","close":"last","volume":"sum"}).dropna()

### 2.5 — Data summary & quality visualization

In [ ]:
print(f"\n{'='*60}")
print(f"DATA SUMMARY")
print(f"{'='*60}")
print(f"Total bars:     {len(df):,}")
print(f"Date range:     {df.index[0]} → {df.index[-1]}")
print(f"Trading days:   {(df.index[-1] - df.index[0]).days}")
print(f"Bars per day:   {len(df) / max((df.index[-1] - df.index[0]).days, 1):.0f}")
print(f"Price range:    {df['close'].min():.2f} — {df['close'].max():.2f}")
print(f"Volatility (ann): {df['close'].pct_change().std() * np.sqrt(252*24*60):.1%}")
print(f"NaN count:      {df.isna().sum().sum()}")
print(f"Duplicates:     {df.index.duplicated().sum()}")

# Plot close price and volume
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
ax1.plot(df.index, df['close'], linewidth=0.5, color='#1f77b4')
ax1.set_ylabel('Close Price')
ax1.set_title('Price & Volume Overview')
ax1.grid(True, alpha=0.3)
ax2.bar(df.index, df['volume'], width=0.001, color='#ff7f0e', alpha=0.7)
ax2.set_ylabel('Volume')
ax2.set_xlabel('Date')
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Data quality & validation

Run the production-grade validation pipeline. Flags schema issues, OHLC violations, gaps, and outliers.

In [ ]:
from services.training_pipeline import DataValidator

# Schema & completeness check
validation = DataValidator.validate_all(df, min_bars=1000)
print("Data Validation Results:")
print(f"  Schema valid:      {validation['schema_valid']}")
print(f"  Types valid:       {validation['types_valid']}")
print(f"  OHLC valid:        {validation['ohlc_valid']}")
print(f"  Completeness:      {validation['completeness_valid']}")
print(f"  Overall valid:     {validation['overall_valid']}")
if validation['errors']:
    print(f"  Errors:")
    for e in validation['errors']:
        print(f"    - {e}")

if not validation['overall_valid']:
    raise ValueError(f"Data validation failed: {validation['errors']}")

# Clean and check
cleaner = DataCleaner()
df_clean = cleaner.remove_duplicates(df)
df_clean = cleaner.fill_gaps(df_clean, max_gap_minutes=5, interval="1m")
df_clean, n_outliers = cleaner.remove_outliers(df_clean, zscore=4.0)
df_clean = cleaner.validate_sessions(df_clean, min_bars=50)

print(f"\nAfter cleaning:")
print(f"  Bars: {len(df):,} → {len(df_clean):,} (-{len(df)-len(df_clean):,})")
print(f"  Outliers removed: {n_outliers}")

# Regime shift detection
n_shifts = cleaner.detect_regime_shift(df_clean)
print(f"  Regime shifts:    {n_shifts}")

# Use cleaned data going forward
df = df_clean
del df_clean

## 4. Smoke test (gate)

Train on the first 70% of a 5–10k-bar slice, evaluate on the last 30%. **If OOS win-rate < 50% we abort** — no point burning hours on full training when the small slice already shows no edge. The threshold is conservative; published 1m FX SOTA is 55–60%. We also compute per-class precision/recall and a confusion matrix.

In [ ]:
# Smoke test config
ASSET = "XAUUSD"
SMOKE_BARS = 8000
SMOKE_TRAIN_PCT = 0.70
SMOKE_WR_THRESHOLD = 0.50
SMOKE_F1_THRESHOLD = 0.18

smoke_df = df.iloc[:SMOKE_BARS].copy() if len(df) > SMOKE_BARS else df.copy()
if smoke_df.index.tz is None:
    smoke_df.index = smoke_df.index.tz_localize("UTC")
print(f"Smoke slice: {len(smoke_df):,} bars · {smoke_df.index[0]} → {smoke_df.index[-1]}")

In [ ]:
# Build features + smoke-train the default XGBoost
sys_smoke = PrecisionTradingSystem(asset=Asset(ASSET), model_type="xgboost", use_hmm=True)
split = int(len(smoke_df) * SMOKE_TRAIN_PCT)
sys_smoke.train(smoke_df.iloc[:split])

# Build OOS features + predict
test_feat = sys_smoke._build_features(smoke_df.iloc[split:])
preds = sys_smoke.signal_model.predict(test_feat)
y_true = triple_barrier_labels_vectorized(
    test_feat,
    pt_atr_mult=getattr(sys_smoke.config, "atr_multiplier_tp1", 1.5),
    sl_atr_mult=getattr(sys_smoke.config, "atr_multiplier_stop", 1.0),
    max_bars=10,
)
y_pred = preds.reindex(y_true.index).fillna(0).values
yt = y_true.values
mask = ~(np.isnan(yt) | np.isnan(y_pred.astype(float)))

# Core metrics
smoke_f1 = f1_score(yt[mask], y_pred[mask], average="macro", zero_division=0)
smoke_acc = (yt[mask] == y_pred[mask]).mean()
nz = y_pred[mask] != 0
smoke_wr = (np.sign(yt[mask][nz]) == np.sign(y_pred[mask][nz])).mean() if nz.sum() else 0.0
smoke_mcc = matthews_corrcoef(yt[mask], y_pred[mask])

print(f"\n── SMOKE RESULTS ──")
print(f"F1 macro:   {smoke_f1:.3f}  (threshold {SMOKE_F1_THRESHOLD:.2f})")
print(f"Accuracy:   {smoke_acc:.1%}")
print(f"Win rate:   {smoke_wr:.1%}  (threshold {SMOKE_WR_THRESHOLD:.0%})")
print(f"MCC:        {smoke_mcc:.3f}")

# Per-class breakdown
print(f"\n── Per-Class Metrics ──")
classes = sorted(set(int(y) for y in np.unique(yt[mask])))
for c in classes:
    cmask = yt[mask] == c
    if cmask.sum() > 5:
        c_acc = (y_pred[mask][cmask] == c).mean()
        c_rec = recall_score(yt[mask] == c, y_pred[mask] == c, zero_division=0)
        c_pre = precision_score(yt[mask] == c, y_pred[mask] == c, zero_division=0)
        direction = {1: 'LONG', -1: 'SHORT', 0: 'HOLD'}.get(c, str(c))
        print(f"  {direction:6s} → prec={c_pre:.3f}  rec={c_rec:.3f}  acc={c_acc:.3f}  n={cmask.sum()}")

print(f"\nPred dist:  {pd.Series(y_pred[mask]).value_counts().sort_index().to_dict()}")
print(f"True dist:  {pd.Series(yt[mask]).value_counts().sort_index().to_dict()}")

# Confusion matrix
cm = confusion_matrix(yt[mask], y_pred[mask], labels=classes)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['SHORT','HOLD','LONG'],
            yticklabels=['SHORT','HOLD','LONG'])
plt.title('Smoke Test — Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

In [ ]:
# ── GATE: abort if smoke fails ──
PROCEED = (smoke_wr >= SMOKE_WR_THRESHOLD) and (smoke_f1 >= SMOKE_F1_THRESHOLD)

if not PROCEED:
    print("\n❌ SMOKE FAILED — not proceeding to full training.")
    print("Diagnostic suggestions:")
    print("  1. More data — 1m needs 60+ days minimum")
    print("  2. Different max_bars — try 5 (faster) or 20 (longer)")
    print("  3. Check for timezone misalignment")
    print("  4. Try BTC over XAU — crypto has more microstructure edge")
    print("  5. Run Dukascopy for years of 1m history")
    raise SystemExit("Smoke gate failed")

print(f"\n✅ SMOKE PASSED — WR={smoke_wr:.1%} F1={smoke_f1:.3f} → proceeding to full training.")

## 5. Walk-forward cross-validation

Purged walk-forward CV (López de Prado, 2018) with embargo. Tests temporal stability — if OOS metrics are wildly inconsistent across folds, the edge may be spurious.

In [ ]:
if df.index.tz is None:
    df.index = df.index.tz_localize("UTC")

sys_cv = PrecisionTradingSystem(asset=Asset(ASSET), model_type="xgboost", use_hmm=True)

# Run walk-forward backtest with 3 folds
print("Running purged walk-forward CV (3 folds)…")
cv_results = sys_cv.walk_forward_backtest(df, n_splits=3, test_size_pct=0.10)

print(f"\n── Walk-Forward CV Summary ──")
print(f"Folds:            {cv_results['n_splits']}")
print(f"Total Trades:     {cv_results['total_trades']}")
print(f"Win Rate:         {cv_results['win_rate']:.1%}")
print(f"F1 Macro:         {cv_results['f1_macro']:.3f}")
print(f"MCC:              {cv_results['mcc']:.3f}")
print(f"Sharpe:           {cv_results['sharpe']:.2f}")
print(f"Max Drawdown:     {cv_results['max_drawdown']:.2f}")
print(f"Total Return:     {cv_results['total_return']:.1%}")
print(f"ECE:              {cv_results['ece']:.4f}")
if cv_results.get('p_value') is not None:
    print(f"P-value:          {cv_results['p_value']:.4f} {'**' if cv_results['p_value'] < 0.05 else ''}")

# Per-fold breakdown
if cv_results.get('per_fold'):
    print(f"\n── Per-Fold Metrics ──")
    fold_metrics = pd.DataFrame(cv_results['per_fold'])
    print(fold_metrics[['fold','trades','win_rate','f1_macro','sharpe']].to_string(index=False))

## 6. Full training — CatBoost + LightGBM + XGBoost stacking ensemble

**Stacking architecture (CPU-friendly, no GPU needed):**
1. **CatBoost** on tabular microstructure features (handles categoricals natively).
2. **LightGBM** as a parallel base learner.
3. **XGBoost** as the third base learner.
4. **Logistic regression meta-learner** stacks the three base models' probabilities.
5. **Isotonic calibration** wraps the final stacked output.

Trained with **temporal-decay sample weights** — recent bars receive exponentially higher weight.

In [ ]:
# Run the TrainingPipeline (data clean + split + features + Markov)
if df.index.tz is None:
    df.index = df.index.tz_localize("UTC")

def fetch_fn(start, end, interval="1m"):
    s = pd.Timestamp(start, tz="UTC") if not hasattr(start, "tz") or start.tz is None else start
    e = pd.Timestamp(end, tz="UTC") if not hasattr(end, "tz") or end.tz is None else end
    return df.loc[(df.index >= s) & (df.index <= e)]

total_days = max((df.index[-1] - df.index[0]).days, 7)
test_days = max(5, total_days // 10)
val_days = max(3, total_days // 14)
train_days = total_days - test_days - val_days - 1
print(f"Window: train={train_days}d val={val_days}d test={test_days}d")

cfg = TrainingConfig(
    asset=ASSET, model_type="xgboost",
    train_days=train_days, validation_days=val_days, test_days=test_days,
    interval="1m" if (df.index[1] - df.index[0]).total_seconds() < 600 else "1h",
    min_bars_per_day=20,
    temporal_decay_halflife=720,
    enable_hyperopt=False,
    enable_drift_detection=False,
)
sys_full = PrecisionTradingSystem(asset=Asset(ASSET), model_type="xgboost", use_hmm=True)
pipeline = TrainingPipeline(sys_full, cfg)
metrics_xgb = pipeline.run(fetch_fn)
print(f"\n── XGBoost baseline (TrainingPipeline) ──")
print(f"  Train F1={metrics_xgb.train_f1:.3f}  Val F1={metrics_xgb.val_f1:.3f}  OOS F1={metrics_xgb.oos_f1:.3f}")
print(f"  OOS WR={metrics_xgb.oos_winrate:.1%}  OOS Acc={metrics_xgb.oos_accuracy:.1%}  OOS MCC={metrics_xgb.oos_mcc:.3f}")
print(f"  Saved: {metrics_xgb.model_path}")

In [ ]:
# Build the stacking ensemble on top.
# Reuse the engineered features by re-running _build_features on full df.
feats_full = sys_full._build_features(df)
feature_cols = [c for c in feats_full.columns
                if c not in ["open", "high", "low", "close", "volume"]]
X_all = feats_full[feature_cols].fillna(0).values
y_all = triple_barrier_labels_vectorized(
    feats_full,
    pt_atr_mult=getattr(sys_full.config, "atr_multiplier_tp1", 1.5),
    sl_atr_mult=getattr(sys_full.config, "atr_multiplier_stop", 1.0),
    max_bars=10,
).values
n = len(X_all)
test_n = max(50, int(n * test_days / total_days))
val_n = max(50, int(n * val_days / total_days))
X_train, y_train = X_all[: -(test_n + val_n)], y_all[: -(test_n + val_n)]
X_val,   y_val   = X_all[-(test_n + val_n) : -test_n], y_all[-(test_n + val_n) : -test_n]
X_test,  y_test  = X_all[-test_n :], y_all[-test_n :]
print(f"Train/Val/Test: {len(X_train)} / {len(X_val)} / {len(X_test)}")

# Encode labels {-1, 0, +1} → {0, 1, 2}
label_map = {-1: 0, 0: 1, 1: 2}
inv_map = {v: k for k, v in label_map.items()}
y_train_e = np.array([label_map.get(int(y), 1) for y in y_train])
y_val_e = np.array([label_map.get(int(y), 1) for y in y_val])
y_test_e = np.array([label_map.get(int(y), 1) for y in y_test])

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

print(f"X_train_s shape: {X_train_s.shape}, n_features: {len(feature_cols)}")

In [ ]:
# ── Train base learners with temporal decay sample weights ──
halflife = 720
decay_lambda = np.log(2) / halflife
t = np.arange(len(X_train))
weights = np.exp(decay_lambda * (t - len(X_train)))
weights = weights / weights.sum() * len(X_train)

base_preds_val = []
base_preds_test = []
base_models = {}
base_train_scores = {}

# 1) XGBoost
print("Training XGBoost…")
xgb_clf = xgb.XGBClassifier(
    n_estimators=2000, max_depth=6, learning_rate=0.01,
    reg_lambda=5.0, min_child_weight=20,
    subsample=0.9, colsample_bytree=0.9,
    tree_method="hist", random_state=42,
    early_stopping_rounds=100, eval_metric="mlogloss",
)
xgb_clf.fit(X_train_s, y_train_e, sample_weight=weights,
            eval_set=[(X_val_s, y_val_e)], verbose=False)
base_preds_val.append(xgb_clf.predict_proba(X_val_s))
base_preds_test.append(xgb_clf.predict_proba(X_test_s))
base_models["xgb"] = xgb_clf
base_train_scores["xgb"] = (xgb_clf.predict(X_val_s) == y_val_e).mean()

# 2) LightGBM
print("Training LightGBM…")
lgb_clf = lgb.LGBMClassifier(
    n_estimators=2000, max_depth=8, learning_rate=0.01,
    reg_lambda=5.0, min_child_samples=30,
    subsample=0.9, colsample_bytree=0.9,
    random_state=42, verbose=-1, n_jobs=-1,
)
lgb_clf.fit(X_train_s, y_train_e, sample_weight=weights,
            eval_set=[(X_val_s, y_val_e)],
            callbacks=[lgb.early_stopping(100, verbose=False)])
base_preds_val.append(lgb_clf.predict_proba(X_val_s))
base_preds_test.append(lgb_clf.predict_proba(X_test_s))
base_models["lgb"] = lgb_clf
base_train_scores["lgb"] = (lgb_clf.predict(X_val_s) == y_val_e).mean()

# 3) CatBoost (if available)
if HAS_CATBOOST:
    print("Training CatBoost…")
    cb_clf = CatBoostClassifier(
        iterations=2000, depth=6, learning_rate=0.01,
        l2_leaf_reg=5.0, random_seed=42, verbose=False,
        early_stopping_rounds=100,
    )
    cb_clf.fit(X_train_s, y_train_e, sample_weight=weights,
               eval_set=(X_val_s, y_val_e), use_best_model=True)
    base_preds_val.append(cb_clf.predict_proba(X_val_s))
    base_preds_test.append(cb_clf.predict_proba(X_test_s))
    base_models["cat"] = cb_clf
    base_train_scores["cat"] = (cb_clf.predict(X_val_s) == y_val_e).mean()

print(f"\n{len(base_models)} base learners trained.")
for name, score in base_train_scores.items():
    print(f"  {name}: val_acc={score:.3f}")

# Feature importance comparison
fig, axes = plt.subplots(len(base_models), 1, figsize=(12, 3*len(base_models)))
if len(base_models) == 1:
    axes = [axes]
for ax, (name, m) in zip(axes, base_models.items()):
    if hasattr(m, 'feature_importances_'):
        importances = pd.Series(m.feature_importances_, index=feature_cols[:len(m.feature_importances_)])
        top = importances.nlargest(15)
        ax.barh(range(len(top)), top.values)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top.index, fontsize=8)
        ax.set_title(f'{name} — Top 15 Features')
        ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 6.5 — Stacking meta-learner & calibration

In [ ]:
# ── Stacking meta-learner: logistic regression on base-learner probabilities ──
M_train_meta = np.hstack(base_preds_val)
M_test_meta = np.hstack(base_preds_test)
print(f"Meta-features: {M_train_meta.shape[1]} dims (n_bases × n_classes)")

meta = LogisticRegression(
    multi_class="multinomial", solver="lbfgs",
    C=1.0, max_iter=200, random_state=42,
)
meta.fit(M_train_meta, y_val_e)

# ── Isotonic calibration on top of meta ──
calib_meta = CalibratedClassifierCV(meta, method="isotonic", cv="prefit")
calib_meta.fit(M_train_meta, y_val_e)

# Final predictions on test set
y_pred_test_e = calib_meta.predict(M_test_meta)
y_pred_test = np.array([inv_map[int(p)] for p in y_pred_test_e])
y_test_orig = np.array([inv_map[int(t)] for t in y_test_e])

stack_f1 = f1_score(y_test_orig, y_pred_test, average="macro", zero_division=0)
stack_acc = (y_test_orig == y_pred_test).mean()
nz = y_pred_test != 0
stack_wr = (np.sign(y_test_orig[nz]) == np.sign(y_pred_test[nz])).mean() if nz.sum() else 0.0
stack_mcc = matthews_corrcoef(y_test_orig, y_pred_test)

# Per-class
stack_long_prec = precision_score(y_test_orig == 1, y_pred_test == 1, zero_division=0)
stack_short_prec = precision_score(y_test_orig == -1, y_pred_test == -1, zero_division=0)
stack_long_rec = recall_score(y_test_orig == 1, y_pred_test == 1, zero_division=0)
stack_short_rec = recall_score(y_test_orig == -1, y_pred_test == -1, zero_division=0)

print(f"\n── STACKED ENSEMBLE — OOS RESULTS ──")
print(f"F1 macro:    {stack_f1:.3f}")
print(f"Accuracy:    {stack_acc:.1%}")
print(f"Win rate:    {stack_wr:.1%}")
print(f"MCC:         {stack_mcc:.3f}")
print(f"Long prec:   {stack_long_prec:.3f}  rec: {stack_long_rec:.3f}")
print(f"Short prec:  {stack_short_prec:.3f}  rec: {stack_short_rec:.3f}")
print(f"Pred dist:   {pd.Series(y_pred_test).value_counts().sort_index().to_dict()}")
print(f"True dist:   {pd.Series(y_test_orig).value_counts().sort_index().to_dict()}")

## 7. Calibration verification

Check if the calibrated meta-learner produces well-calibrated probabilities (reliability diagram). Poor calibration = overconfident predictions = money-losing.

In [ ]:
# Reliability diagrams — before vs after calibration
from sklearn.calibration import calibration_curve

proba_before = meta.predict_proba(M_test_meta)
proba_after = calib_meta.predict_proba(M_test_meta)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
class_labels = ['SHORT', 'HOLD', 'LONG']

for i, (ax, label) in enumerate(zip(axes, class_labels)):
    # Before calibration
    frac_before, mean_before = calibration_curve(
        y_test_e == i, proba_before[:, i], n_bins=10, strategy='uniform'
    )
    # After calibration
    frac_after, mean_after = calibration_curve(
        y_test_e == i, proba_after[:, i], n_bins=10, strategy='uniform'
    )
    ax.plot([0, 1], [0, 1], 'k--', linewidth=0.5, label='Perfect')
    ax.plot(mean_before, frac_before, 's-', label='Before calib', markersize=5)
    ax.plot(mean_after, frac_after, 'o-', label='After calib', markersize=5)
    ax.set_title(f'{label} Calibration')
    ax.set_xlabel('Predicted Probability')
    ax.set_ylabel('Fraction of Positives')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Reliability Diagrams — Stacked Ensemble')
plt.tight_layout()
plt.show()

# Expected Calibration Error
def compute_ece(probas, labels, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (probas >= bins[i]) & (probas < bins[i + 1])
        if mask.sum() == 0:
            continue
        acc = labels[mask].mean()
        conf = probas[mask].mean()
        ece += (mask.sum() / len(probas)) * abs(acc - conf)
    return ece

for i, label in enumerate(class_labels):
    ece_before = compute_ece(proba_before[:, i], (y_test_e == i).astype(float))
    ece_after = compute_ece(proba_after[:, i], (y_test_e == i).astype(float))
    print(f"{label}: ECE before={ece_before:.4f} after={ece_after:.4f}")

## 8. Risk-adjusted performance evaluation

Compute Sharpe ratio, max drawdown, profit factor, and trade distribution from test-set trades.

In [ ]:
# Simulate trades on test set using the stacked ensemble predictions
rm = RiskManager(sys_full.config)

for i in range(len(y_pred_test)):
    sig = y_pred_test[i]
    if sig == 1:
        direction = TradeDirection.LONG
    elif sig == -1:
        direction = TradeDirection.SHORT
    else:
        continue
    
    # Use the bar at test index for price info
    bar_idx = y_all[-(test_n) + i] if -(test_n) + i < len(y_all) else 0
    row = feats_full.iloc[-(test_n) + i] if -(test_n) + i < len(feats_full) else feats_full.iloc[-1]
    atr = float(row.get("atr_14", 1.0))
    entry = float(row.get("close", feats_full.iloc[-(test_n) + i]["close"]))
    
    rm.open_trade(
        ts=feats_full.index[-(test_n) + i],
        direction=direction,
        entry=entry,
        atr=atr,
        use_kelly=False
    )

rm.close_all(feats_full.index[-1], float(feats_full["close"].iloc[-1]))
trade_stats = rm.get_stats()

print(f"\n── Risk-Adjusted Performance (Test Set) ──")
print(f"Total Trades:    {trade_stats['total_trades']}")
print(f"Win Rate:        {trade_stats['win_rate']:.1%}")
print(f"Long Prec:       {trade_stats['precision_long']:.3f}")
print(f"Short Prec:      {trade_stats['precision_short']:.3f}")
print(f"Profit Factor:   {trade_stats['profit_factor']:.2f}")
print(f"Sharpe Ratio:    {trade_stats['sharpe']:.2f}")
print(f"Max Drawdown:    {trade_stats['max_drawdown']:.2f}")
print(f"Total PnL:       {trade_stats['total_pnl']:.2f}")

# Equity curve
if trade_stats.get('equity_curve'):
    eq = trade_stats['equity_curve']
    plt.figure(figsize=(12, 4))
    plt.plot(eq, linewidth=1, color='#2ca02c')
    plt.axhline(y=eq[0], color='gray', linestyle='--', linewidth=0.5, label='Initial')
    plt.title('Equity Curve — Test Set')
    plt.ylabel('Equity')
    plt.xlabel('Trade #')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 9. Comparison: stacked vs single XGBoost

If stacking doesn't beat single XGBoost by ≥1pp on win-rate, ship the simpler model (Occam's razor).

In [ ]:
# Plain XGBoost baseline metrics on the same test slice
y_pred_xgb_e = xgb_clf.predict(X_test_s)
y_pred_xgb = np.array([inv_map[int(p)] for p in y_pred_xgb_e])
xgb_f1 = f1_score(y_test_orig, y_pred_xgb, average="macro", zero_division=0)
xgb_acc = (y_test_orig == y_pred_xgb).mean()
nz_x = y_pred_xgb != 0
xgb_wr = (np.sign(y_test_orig[nz_x]) == np.sign(y_pred_xgb[nz_x])).mean() if nz_x.sum() else 0.0
xgb_mcc = matthews_corrcoef(y_test_orig, y_pred_xgb)

print("            F1     Acc     WR      MCC     LongPrec  ShortPrec")
print(f"XGBoost   {xgb_f1:.3f}  {xgb_acc:.1%}  {xgb_wr:.1%}  {xgb_mcc:.3f}  "
      f"{precision_score(y_test_orig==1, y_pred_xgb==1, zero_division=0):.3f}    "
      f"{precision_score(y_test_orig==-1, y_pred_xgb==-1, zero_division=0):.3f}")
print(f"Stacked   {stack_f1:.3f}  {stack_acc:.1%}  {stack_wr:.1%}  {stack_mcc:.3f}  "
      f"{stack_long_prec:.3f}    {stack_short_prec:.3f}")

wr_lift = stack_wr - xgb_wr
f1_lift = stack_f1 - xgb_f1
print(f"\nStacking lift: WR {wr_lift:+.1%}, F1 {f1_lift:+.3f}")

USE_STACK = wr_lift >= 0.01 and f1_lift >= 0
FINAL_MODEL = "stacked_ensemble" if USE_STACK else "xgboost_only"
print(f"→ Recommend: {FINAL_MODEL}")
if not USE_STACK:
    print(f"  (Stacking didn't beat XGBoost by ≥1pp WR — shipping simpler model)")

# Statistical significance test (McNemar)
from scipy.stats import binomtest
correct_xgb = (y_test_orig == y_pred_xgb).astype(int)
correct_stack = (y_test_orig == y_pred_test).astype(int)
n_xgb_only = ((correct_xgb == 1) & (correct_stack == 0)).sum()
n_stack_only = ((correct_xgb == 0) & (correct_stack == 1)).sum()
if n_xgb_only + n_stack_only > 0:
    p_val = binomtest(n_stack_only, n_xgb_only + n_stack_only, 0.5).pvalue
    print(f"McNemar p-value: {p_val:.4f} {'(significant)' if p_val < 0.05 else '(not significant)'}")

# Confusion matrices side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
cm_xgb = confusion_matrix(y_test_orig, y_pred_xgb, labels=[-1, 0, 1])
cm_stack = confusion_matrix(y_test_orig, y_pred_test, labels=[-1, 0, 1])
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=['SHORT','HOLD','LONG'], yticklabels=['SHORT','HOLD','LONG'])
ax1.set_title('XGBoost Only')
sns.heatmap(cm_stack, annot=True, fmt='d', cmap='Greens', ax=ax2,
            xticklabels=['SHORT','HOLD','LONG'], yticklabels=['SHORT','HOLD','LONG'])
ax2.set_title('Stacked Ensemble')
plt.suptitle('Confusion Matrix Comparison')
plt.tight_layout()
plt.show()

## 10. Save + download the trained artifact

Saves a single bundle (`*.pkl`) containing all base models, scaler, meta-learner, calibrator, label map, and feature columns.

In [ ]:
import joblib, json
from datetime import datetime

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out_dir = f"/content/precision_models_{ASSET}"
os.makedirs(out_dir, exist_ok=True)
model_path = f"{out_dir}/precision_{ASSET}_{FINAL_MODEL}_{ts}.pkl"

bundle = {
    "asset": ASSET,
    "final_model": FINAL_MODEL,
    "feature_cols": feature_cols,
    "scaler": scaler,
    "label_map": label_map,
    "inv_map": inv_map,
    "base_models": base_models if USE_STACK else None,
    "meta": meta if USE_STACK else None,
    "calibrated_meta": calib_meta if USE_STACK else None,
    "single_xgb": xgb_clf if not USE_STACK else None,
    "trained_at": datetime.now().isoformat(),
    "metrics_xgb_only": {"f1": float(xgb_f1), "acc": float(xgb_acc),
                         "wr": float(xgb_wr), "mcc": float(xgb_mcc)},
    "metrics_stacked": {"f1": float(stack_f1), "acc": float(stack_acc),
                        "wr": float(stack_wr), "mcc": float(stack_mcc),
                        "long_prec": float(stack_long_prec),
                        "short_prec": float(stack_short_prec)} if USE_STACK else None,
    "label_horizon_bars": 10,
    "data_range": [str(df.index[0]), str(df.index[-1])],
    "n_train": int(len(X_train)), "n_val": int(len(X_val)), "n_test": int(len(X_test)),
}
joblib.dump(bundle, model_path, compress=3)

meta_path = model_path.replace(".pkl", "_meta.json")
with open(meta_path, "w") as f:
    json.dump({k: v for k, v in bundle.items()
                if k not in ("scaler", "base_models", "meta",
                              "calibrated_meta", "single_xgb")}, f, indent=2, default=str)

print(f"Saved bundle: {model_path}")
print(f"Saved meta:   {meta_path}")
size_mb = os.path.getsize(model_path) / 1024 / 1024
print(f"Bundle size: {size_mb:.1f} MB")

In [ ]:
# Download the bundle locally
from google.colab import files
files.download(model_path)
files.download(meta_path)

## 11. In-notebook test on the saved bundle

Load the bundle back and run inference on the last few hundred bars to confirm it deserialises correctly.

In [ ]:
loaded = joblib.load(model_path)

# Predict on the last 200 bars
tail = df.tail(200)
tail_feat = sys_full._build_features(tail)
X_tail = scaler.transform(tail_feat[loaded["feature_cols"]].fillna(0).values)

if loaded["final_model"] == "stacked_ensemble":
    base_probs = []
    for name, m in loaded["base_models"].items():
        base_probs.append(m.predict_proba(X_tail))
    M_tail = np.hstack(base_probs)
    enc_preds = loaded["calibrated_meta"].predict(M_tail)
else:
    enc_preds = loaded["single_xgb"].predict(X_tail)

preds = np.array([loaded["inv_map"][int(p)] for p in enc_preds])
print("Tail predictions (last 200 bars):")
print(pd.Series(preds).value_counts().sort_index().to_dict())
print("Last 20 predictions:", preds[-20:].tolist())

# Verify output shape matches
print(f"\nExpected: {len(tail)} bars → {len(preds)} predictions")
assert len(preds) == len(X_tail), "Prediction count mismatch!"
print("✓ Bundle loads and predicts correctly")

## 12. (Optional) Drop the bundle into the local repo

On your local machine after downloading:

```bash
cp ~/Downloads/precision_XAUUSD_*.pkl trading_terminal/frontend/data/models/XAUUSD/
```

Then load it via the ModelRegistry or directly with joblib. For production, promote the best model:

```python
from services.training_pipeline import ModelRegistry
registry = ModelRegistry("data/model_registry")
registry.register_model("XAUUSD_stacked", model_path, metrics, config)
registry.promote_model(model_id, "production")
```

## What to do if metrics still aren't where you want

1. **More data.** 30 days of yfinance 1m on FX is too little. Use the prebuilt `data/historical/<PAIR>_1m.csv` (60 days) or Dukascopy (years).
2. **Different label horizon.** Try `max_bars=5` (faster signals) or `max_bars=20` (longer holds). The optimum is asset-dependent.
3. **Different barriers.** Symmetric `pt=sl=1.5×ATR` is balanced; if shorts dominate, increase `sl_mult`.
4. **Cost-aware labels.** Add `cost_pct=0.0002` (gold) / `0.0008` (BTC) to penalise tight wins that costs eat.
5. **Skip the BiLSTM.** Tabular boosted trees consistently beat sequence models on 1m FX/gold (Borrageiro 2022, Tashiro 2023). More data > more model complexity.
6. **Feature ablation.** Run with subsets of features (microstructure only, price only, flow only) to find what drives signal.
7. **Different assets.** BTC/USD typically has stronger 1m microstructure edge than XAU/USD. EUR/USD has the weakest.
8. **Regime-specific models.** Train separate models per HMM regime and route at inference time.